# Convert TIFF experiment to OME-Zarr v0.5

Demonstrates how to migrate an existing experiment stored in the old
TIFF-per-frame format to the new OME-Zarr v0.5 format using
`convert_tiff_to_omezarr`.

**Old format** (one TIFF per frame, subfolders for data types):
```
experiment/
├── raw/            000_00000.tiff  (C, H, W)
├── labels/         000_00000.tiff
├── particles/      000_00000.tiff
├── stim_mask/      000_00000.tiff
├── ref/            000_00002.tiff
└── tracks/         0_latest.parquet
```

**New format** (single OME-Zarr store):
```
output/
├── acquisition.ome.zarr/
│   ├── 0/              raw (t, p, c, y, x)
│   └── labels/
│       ├── labels/0    segmentation masks
│       ├── particles/0 tracked labels
│       └── stim_mask/0 stimulation masks
├── ref/                TIFF fallback
└── tracks/             parquet (copied)
```

## 1. Generate sample old-format data

First, let's create a synthetic experiment in the old TIFF format
so we have something to convert. (Skip this section if you already
have real experiment data.)

In [33]:
import os
import shutil
import numpy as np
import pandas as pd
import tifffile

# Create a temporary old-format experiment
src_path = "./sample_old_experiment"
if os.path.exists(src_path):
    shutil.rmtree(src_path)

N_FOV = 2
N_TIMEPOINTS = 5
N_CHANNELS = 2
IMG_H, IMG_W = 128, 128

# Create subfolders
for folder in ["raw", "labels", "particles", "stim_mask", "ref", "tracks"]:
    os.makedirs(os.path.join(src_path, folder), exist_ok=True)

rng = np.random.default_rng(42)

for fov in range(N_FOV):
    rows = []
    for t in range(N_TIMEPOINTS):
        fname = f"{fov:03d}_{t:05d}"

        # Raw: (C, H, W) uint16
        raw = rng.integers(0, 4096, (N_CHANNELS, IMG_H, IMG_W), dtype=np.uint16)
        tifffile.imwrite(os.path.join(src_path, "raw", fname + ".tiff"), raw)

        # Labels: (H, W) uint16 — simple grid segmentation
        labels = np.zeros((IMG_H, IMG_W), dtype=np.uint16)
        cell_id = 1
        for yi in range(0, IMG_H, 32):
            for xi in range(0, IMG_W, 32):
                labels[yi : yi + 28, xi : xi + 28] = cell_id
                cell_id += 1
        tifffile.imwrite(os.path.join(src_path, "labels", fname + ".tiff"), labels)

        # Particles: same as labels but with shifted IDs (simulating tracking)
        particles = (labels + fov * 100).astype(np.uint16)
        particles[labels == 0] = 0
        tifffile.imwrite(
            os.path.join(src_path, "particles", fname + ".tiff"), particles
        )

        # Stim mask: binary (H, W) uint8
        stim_mask = np.zeros((IMG_H, IMG_W), dtype=np.uint8)
        if t % 2 == 1:  # stimulate on odd frames
            stim_mask[32:96, 32:96] = 255
        tifffile.imwrite(
            os.path.join(src_path, "stim_mask", fname + ".tiff"), stim_mask
        )

        # Tracking data
        for lbl in range(1, cell_id):
            rows.append(
                {
                    "timestep": t,
                    "fov": fov,
                    "particle": lbl,
                    "label": lbl,
                    "x": float(rng.uniform(0, IMG_W)),
                    "y": float(rng.uniform(0, IMG_H)),
                }
            )

    # Ref image: only at last timestep
    ref_img = rng.integers(0, 4096, (IMG_H, IMG_W), dtype=np.uint16)
    ref_fname = f"{fov:03d}_{N_TIMEPOINTS - 1:05d}"
    tifffile.imwrite(os.path.join(src_path, "ref", ref_fname + ".tiff"), ref_img)

    # Save tracking parquet
    df = pd.DataFrame(rows)
    df.to_parquet(os.path.join(src_path, "tracks", f"{fov}_latest.parquet"))

print("Old-format experiment created:")
for folder in sorted(os.listdir(src_path)):
    folder_path = os.path.join(src_path, folder)
    if os.path.isdir(folder_path):
        n_files = len(os.listdir(folder_path))
        print(f"  {folder}/  ({n_files} files)")

Old-format experiment created:
  labels/  (10 files)
  particles/  (10 files)
  raw/  (10 files)
  ref/  (2 files)
  stim_mask/  (10 files)
  tracks/  (2 files)


## 2. Convert to OME-Zarr

In [34]:
from rtm_pymmcore.core.conversion import convert_tiff_to_omezarr

dst_path = "./sample_omezarr_output"

zarr_path = convert_tiff_to_omezarr(
    src_path=src_path,
    dst_path=dst_path,
    channel_names=["DAPI", "FITC"],  # name the raw channels
    label_chunk_t=1,
    label_shard_t=50,
    overwrite=True,
)

Source: ./sample_old_experiment
  FOVs: 2, Timepoints: 5, Channels: 2 ['DAPI', 'FITC']
  Image size: 128x128, dtype: uint16
  Raw frames: 10
  Label folders: ['labels', 'particles', 'stim_mask']
  Ref folders: ['ref']

Writing raw frames...
  Wrote 10 raw frames
Writing label 'labels' (10 frames)...
Writing label 'particles' (10 frames)...
Writing label 'stim_mask' (10 frames)...
Writing ref 'ref' (2 frames as TIFF)...

OME-Zarr store created: sample_omezarr_output\acquisition.ome.zarr
Copied tracks/ (2 parquet files)
Conversion complete.


## 3. Inspect the OME-Zarr store

In [35]:
import zarr

store = zarr.open(zarr_path, mode="r")
print("Zarr store tree:")
print(store.tree())

Zarr store tree:


/
├── 0 (5, 2, 2, 128, 128) uint16
└── labels
    ├── labels
    │   └── 0 (5, 2, 128, 128) uint16
    ├── particles
    │   └── 0 (5, 2, 128, 128) uint16
    └── stim_mask
        └── 0 (5, 2, 128, 128) uint8

In [36]:
# Raw array shape and metadata
raw = store["0"]
print(f"Raw array: shape={raw.shape}, dtype={raw.dtype}")

ome = store.attrs.get("ome", {})
axes = ome.get("multiscales", [{}])[0].get("axes", [])
print(f"Axes: {[a['name'] for a in axes]}")

channels = ome.get("omero", {}).get("channels", [])
print(f"Channels: {[ch['label'] for ch in channels]}")

Raw array: shape=(5, 2, 2, 128, 128), dtype=uint16
Axes: ['t', 'p', 'c', 'y', 'x']
Channels: ['DAPI', 'FITC']


In [37]:
# Label arrays
labels_grp = store["labels"]
label_names = labels_grp.attrs.get("labels", [])
print(f"Label groups: {label_names}")

for name in label_names:
    arr = store[f"labels/{name}/0"]
    print(f"  {name}: shape={arr.shape}, dtype={arr.dtype}")

Label groups: ['labels', 'particles', 'stim_mask']
  labels: shape=(5, 2, 128, 128), dtype=uint16
  particles: shape=(5, 2, 128, 128), dtype=uint16
  stim_mask: shape=(5, 2, 128, 128), dtype=uint8


## 4. Verify data integrity

Compare a few frames from the old TIFFs against the new OME-Zarr.

In [38]:
raw_zarr = store["0"]
labels_zarr = store["labels/labels/0"]
particles_zarr = store["labels/particles/0"]
stim_mask_zarr = store["labels/stim_mask/0"]

n_fov = raw_zarr.shape[1] if raw_zarr.ndim == 5 else 1
print(f"Verifying {N_FOV} FOVs x {N_TIMEPOINTS} timepoints...\n")

all_ok = True
for fov in range(N_FOV):
    for t in range(N_TIMEPOINTS):
        fname = f"{fov:03d}_{t:05d}"

        # Raw
        old_raw = tifffile.imread(os.path.join(src_path, "raw", fname + ".tiff"))
        if n_fov > 1:
            new_raw = np.array(raw_zarr[t, fov, :, :, :])
        else:
            new_raw = np.array(raw_zarr[t, :, :, :])
        if not np.array_equal(old_raw, new_raw):
            print(f"  MISMATCH raw fov={fov} t={t}")
            all_ok = False

        # Labels
        old_lbl = tifffile.imread(os.path.join(src_path, "labels", fname + ".tiff"))
        if n_fov > 1:
            new_lbl = np.array(labels_zarr[t, fov, :, :])
        else:
            new_lbl = np.array(labels_zarr[t, :, :])
        if not np.array_equal(old_lbl, new_lbl):
            print(f"  MISMATCH labels fov={fov} t={t}")
            all_ok = False

        # Particles
        old_part = tifffile.imread(os.path.join(src_path, "particles", fname + ".tiff"))
        if n_fov > 1:
            new_part = np.array(particles_zarr[t, fov, :, :])
        else:
            new_part = np.array(particles_zarr[t, :, :])
        if not np.array_equal(old_part, new_part):
            print(f"  MISMATCH particles fov={fov} t={t}")
            all_ok = False

        # Stim mask
        old_sm = tifffile.imread(os.path.join(src_path, "stim_mask", fname + ".tiff"))
        if n_fov > 1:
            new_sm = np.array(stim_mask_zarr[t, fov, :, :])
        else:
            new_sm = np.array(stim_mask_zarr[t, :, :])
        if not np.array_equal(old_sm, new_sm):
            print(f"  MISMATCH stim_mask fov={fov} t={t}")
            all_ok = False

if all_ok:
    print("All frames match between old TIFFs and new OME-Zarr!")

Verifying 2 FOVs x 5 timepoints...

All frames match between old TIFFs and new OME-Zarr!


In [28]:
# Verify tracks were copied
for fov in range(N_FOV):
    pq_path = os.path.join(dst_path, "tracks", f"{fov}_latest.parquet")
    assert os.path.exists(pq_path), f"Missing: {pq_path}"
    df = pd.read_parquet(pq_path)
    print(f"FOV {fov}: {len(df)} rows, {df['particle'].nunique()} particles")

print("\nTracks copied successfully.")

FOV 0: 80 rows, 16 particles
FOV 1: 80 rows, 16 particles

Tracks copied successfully.


In [29]:
# Verify ref images were copied as TIFF
ref_dir = os.path.join(dst_path, "ref")
if os.path.exists(ref_dir):
    ref_files = os.listdir(ref_dir)
    print(f"Ref images: {ref_files}")
else:
    print("No ref/ directory (expected if source had none)")

Ref images: ['000_00004.tiff', '001_00004.tiff']


## 5. Convert existing test data

Convert the bundled test data (no labels/tracks, just raw images).

In [ ]:
# Convert the bundled test data with 2 FOVs and optocheck
zarr_path_test = convert_tiff_to_omezarr(
    src_path="../../test_exp_data/01_demo_imgs_w_optocheck",
    dst_path="./test_data_converted",
    channel_names=["miRFP", "mScarlet3"],
)

Source: ../../test_exp_data/01_demo_imgs_w_optocheck
  FOVs: 1, Timepoints: 3, Channels: 2 ['DAPI', 'FITC']
  Image size: 1024x1024, dtype: uint16
  Raw frames: 3
  Ref folders: ['optocheck']

Writing raw frames...
  Wrote 3 raw frames
Writing ref 'optocheck' (1 frames as TIFF)...

OME-Zarr store created: test_data_converted\acquisition.ome.zarr
Conversion complete.


In [31]:
store_test = zarr.open(zarr_path_test, mode="r")
print(store_test.tree())

raw_test = store_test["0"]
print(f"\nRaw: shape={raw_test.shape}, dtype={raw_test.dtype}")

/
└── 0 (3, 2, 1024, 1024) uint16



Raw: shape=(3, 2, 1024, 1024), dtype=uint16


## 6. Cleanup

In [39]:
# Remove generated files
for p in [
    "./sample_old_experiment",
    "./sample_omezarr_output",
    "./test_data_converted",
]:
    if os.path.exists(p):
        shutil.rmtree(p)
        print(f"Removed {p}")

Removed ./sample_old_experiment
Removed ./sample_omezarr_output
Removed ./test_data_converted
